In [1]:
import chipwhisperer as cw
import time
import numpy as np
import struct


scope = cw.scope()
bitfile1 = "/home/40265864@ecit.qub.ac.uk/FFT_Toeplitz/chipwhisperer/firmware/fpgas/aes/vivado/cw305_aes.runs/impl_100t/cw305_top.bit"
bitfile2 = "/home/40265864@ecit.qub.ac.uk/FFT_Toeplitz/cw305_fft/aes/vivado/cw305_aes.runs/impl_100t/cw305_top.bit"
bitfile3 = "/home/40265864@ecit.qub.ac.uk/cw305/chipwhisperer/firmware/fpgas/aes/vivado/cw305_aes.runs/impl_100t/cw305_top.bit"
bitfile4 = "/home/40265864@ecit.qub.ac.uk/PhD/project_TOEPLITZ/FFT_Toeplitz/firmware/fpgas/aes/vivado/cw305_aes.runs/impl_100t/cw305_top.bit"
target = cw.target(scope, cw.targets.CW305, bsfile=bitfile4, force=True, fpga_id='100t', platform='cw305')
scope.default_setup()

(ChipWhisperer Target WARNING|File CW305.py:591) Using default Verilog defines (/home/40265864@ecit.qub.ac.uk/anaconda3/envs/crypto/lib/python3.11/site-packages/chipwhisperer/hardware/firmware/cw305/cw305_aes_defines.v); if this is not what you want, provide them via the defines_files argument


scope.gain.mode                          changed from low                       to high                     
scope.gain.gain                          changed from 0                         to 22                       
scope.gain.db                            changed from 15.0                      to 25.091743119266056       
scope.adc.samples                        changed from 131124                    to 5000                     
scope.clock.clkgen_freq                  changed from 0                         to 7363636.363636363        
scope.clock.adc_freq                     changed from 0                         to 29454545.454545453       
scope.clock.freq_ctr                     changed from 0                         to 10000339                 
scope.io.tio1                            changed from serial_tx                 to serial_rx                
scope.io.tio2                            changed from serial_rx                 to serial_tx                
scope.io.hs2       

In [2]:
scope.adc.samples = 5000
#scope.adc.segments = 12
scope.adc.bits_per_sample = 12
#scope.adc.segment_cycles = 3
scope.adc.offset = 3
scope.adc.stream_mode = True#"segmented"  
#scope.adc.segment_cycle_counter_en = True
scope.adc.basic_mode = "rising_edge"
scope.trigger.triggers = "tio4"
#scope.trigger.module = "basic"
scope.io.tio1 = "serial_rx"
scope.io.tio2 = "serial_tx"
scope.io.hs2 = "disabled"
scope.gain.mode = "high"
target._clksleeptime = 100
scope.gain.gain = 20
scope.adc.timeout = 5  # in milliseconds


In [3]:
target.vccint_set(1.0)
# we only need PLL1:
target.pll.pll_enable_set(True)
target.pll.pll_outenable_set(False, 0)
target.pll.pll_outenable_set(True, 1)
target.pll.pll_outenable_set(False, 2)

# run at 10 MHz:
target.pll.pll_outfreq_set(10E6, 1)

# 1ms is plenty of idling time
target.clkusbautooff = True
target.clksleeptime = 10

scope.clock.clkgen_freq = 40e6
scope.clock.adc_src = 'extclk_x4'
scope.clock.adc_mul = 26
scope.clock.reset_adc()

(ChipWhisperer Scope WARNING|File ChipWhispererHuskyClock.py:1628) scope.clock.adc_src is provided for backwards compability, but scope.clock.clkgen_src and scope.clock.adc_mul should be used for Husky.
(ChipWhisperer Scope WARNING|File ChipWhispererHuskyClock.py:1310) External clock frequency is measured as 10.0 MHz; setting PLL to expect 40.0 MHz, so it may not lock.
(ChipWhisperer Scope WARNING|File ChipWhispererHuskyClock.py:524) ADC frequency must be between 1MHz and 300000000.0MHz - ADC mul has been adjusted to 7
(ChipWhisperer Scope WARNING|File ChipWhispererHuskyClock.py:527) ADC frequency exceeds specification (200 MHz). 
            This may or may not work, depending on temperature, voltage, and luck.
            It may not work reliably.
            You can run scope.adc_test() to check whether ADC data is sampled properly by the FPGA,
            but this doesn't fully verify that the ADC is working properly.
            Set scope.clock.pll._no_warning_freq if you don't wa

In [4]:
def gen_random_input():
    input_text = ["00","00","00","00", "00","00","00","00","00","00","00","00","00","00","00","00",
                "00","00","00","00", "00","00", "00","00", "00","00","00","00","00","00","00","00"]
    
    for i in range(8):
        if np.random.randint(2,size=1)==1:
            input_text[16+(2*i)] = "01"

    data_bytes_input = list([int(byte,16) for byte in reversed(input_text)])

    return data_bytes_input

In [5]:
print(gen_random_input())

[0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


In [7]:
ktp = cw.ktp.Basic()
key, text = ktp.next()
#key = b'\x06\x55\x37'
reg_crypt_textin = ["00","00","00","00", "00","00","00","00","00","00","00","00","00","00","00","00",
                    "00","00","00","00", "00","00", "00","00", "00","00","00","00","00","00","00","00"]




reg_crypt_key1 =  reversed(["00","00","00","00", "00","00","00","00","00","00","00","00","00","00","00","00",
                    "00","00","00","00", "00","00", "00","00", "00","00","00","00","00","00","00","00"])

reg_crypt_key2 =  reversed(["00","00","00","00", "00","00","00","00","00","00","00","00","00","00","00","00",
                    "01","00","00","00", "00","00", "00","00", "00","00","00","00","00","00","00","00"])

reg_crypt_key3 =  reversed(["00","00","00","00", "00","00","00","00","00","00","00","00","00","00","00","00",
                    "00","00","01","00", "00","00", "00","00", "00","00","00","00","00","00","00","00"])

reg_crypt_key4 =  reversed(["00","00","00","00", "00","00","00","00","00","00","00","00","00","00","00","00",
                    "01","00","01","00", "00","00", "00","00", "00","00","00","00","00","00","00","00"])


reg_reset =  ["00","00","00","00", "00","00","00","00","00","00","00","00","00","00","00","00",
                    "00","00","00","00", "00","00", "00","00", "00","00","00","00","00","00","00","00"]


data_bytes_text_in = list([int(byte,16) for byte in reg_crypt_textin])
data_bytes_key_in1 = list([int(byte,16) for byte in reg_crypt_key1])
data_bytes_key_in2 = list([int(byte,16) for byte in reg_crypt_key2])
data_bytes_key_in3 = list([int(byte,16) for byte in reg_crypt_key3])
data_bytes_key_in4 = list([int(byte,16) for byte in reg_crypt_key4])

data_bytes_reset = list([int(byte,16) for byte in reg_reset])

In [8]:
def my_capture(key):
    scope.arm()
    target.fpga_write(target.REG_CRYPT_KEY,key)
    target.fpga_write(target.REG_CRYPT_TEXTIN,key)
    target.go()



    #target.usb_clk_setenabled(False)

    #target.fpga_write(target.REG_USER_LED, [0x01])
    #time.sleep(0.001)

    #target.usb_trigger_toggle()
        
    #time.sleep(target.clksleeptime/1000.0)
    #target.usb_clk_setenabled(True)


    ret = scope.capture(poll_done=True)
    trace = scope.get_last_trace()
    return trace

In [9]:
def align_traces(traces):
    ref_trace = traces[0]  # Use the first trace as reference
    aligned_traces = []

    for trace in traces:
        correlation = np.correlate(trace, ref_trace, mode="valid")  # Compute cross-correlation
        shift = np.argmax(correlation) - (len(trace) - 1)  # Find best alignment
        aligned_trace = np.roll(trace, -shift)  # Shift the trace
        aligned_traces.append(aligned_trace)
    
    return np.array(aligned_traces)

In [25]:
trace = my_capture(data_bytes_key_in4)
oplen = scope.adc.trig_count
print("Operation lenght: %d cycles" % oplen)

Operation lenght: 1540 cycles


In [26]:
traces1 = []
traces2 = []
traces3 = []
traces4 = []

for _ in range(100):
    traces1.append(my_capture(data_bytes_key_in1))
    traces2.append(my_capture(data_bytes_key_in2))
    traces3.append(my_capture(data_bytes_key_in3))
    traces4.append(my_capture(data_bytes_key_in4))



In [27]:
cw.plot(traces1[0]) * cw.plot(traces1[2]) * cw.plot(traces4[0])

:Overlay
   .Curve.I   :Curve   [x]   (y)
   .Curve.II  :Curve   [x]   (y)
   .Curve.III :Curve   [x]   (y)

In [28]:
traces1 = align_traces(traces1)
traces2 = align_traces(traces2)
traces3 = align_traces(traces3)
traces4 = align_traces(traces4)


trace1_avg = np.mean(traces1, axis=0)
trace2_avg = np.mean(traces2, axis=0)
trace3_avg = np.mean(traces3, axis=0)
trace4_avg = np.mean(traces4, axis=0)

In [29]:
cw.plot(traces1[1]) * cw.plot(traces1[5])

:Overlay
   .Curve.I  :Curve   [x]   (y)
   .Curve.II :Curve   [x]   (y)

In [30]:
cw.plot(trace1_avg) * cw.plot(trace2_avg) * cw.plot(trace3_avg) * cw.plot(trace4_avg)

:Overlay
   .Curve.I   :Curve   [x]   (y)
   .Curve.II  :Curve   [x]   (y)
   .Curve.III :Curve   [x]   (y)
   .Curve.IV  :Curve   [x]   (y)

In [ ]:
cw.plot(trace1_avg - traces1[0]) * cw.plot(traces1[3] - traces2[3])

In [ ]:
poi1 = [trace1_avg[490:500],trace1_avg[880:890],trace1_avg[990:1010],trace1_avg[1015:1040],trace1_avg[1570:1580]]
flattened_poi1 = [value for segment in poi1 for value in segment]
poi2 = [trace2_avg[490:500],trace2_avg[880:890],trace2_avg[990:1010],trace2_avg[1015:1040],trace2_avg[1570:1580]]
flattened_poi2 = [value for segment in poi2 for value in segment]
poi3 = [trace3_avg[490:500],trace3_avg[880:890],trace3_avg[990:1010],trace3_avg[1015:1040],trace3_avg[1570:1580]]
flattened_poi3 = [value for segment in poi3 for value in segment]
poi4 = [trace4_avg[490:500],trace4_avg[880:890],trace4_avg[990:1010],trace4_avg[1015:1040],trace4_avg[1570:1580]]
flattened_poi4 = [value for segment in poi4 for value in segment]

In [ ]:
poi_list = []
poi_list.append(np.arange(490,500))
poi_list.append(np.arange(880,890))
poi_list.append(np.arange(990,1010))
poi_list.append(np.arange(1015,1040))
poi_list.append(np.arange(1570,1580))
poi_list = [value for segment in poi_list for value in segment]


In [ ]:
import matplotlib.pyplot as plt
plt.style.use("seaborn-v0_8-darkgrid")
lw = 1.5
plt.plot(flattened_poi1,linewidth=lw,label="00")
plt.plot(flattened_poi2,linewidth=lw,label="01")
plt.plot(flattened_poi3,linewidth=lw,label="10")
plt.plot(flattened_poi4,linewidth=lw,label="11")
plt.legend(loc='upper center', ncol=4, fontsize=12)
plt.xlabel("Sample Point",fontsize=12)
plt.ylabel("Voltage, mV",fontsize=12)
plt.tight_layout()
plt.savefig("/home/40265864@ecit.qub.ac.uk/PhD/project_TOEPLITZ/FPGA_results.png",dpi=500)
plt.show()

In [ ]:
cw.plot(flattened_poi1) * cw.plot(flattened_poi2) * cw.plot(flattened_poi3) * cw.plot(flattened_poi4)

In [ ]:
def extract_poi(poi_list, trace):
    op_trace = []
    poi_ctr = 0
    poi_max = len(poi_list)
    for i in range(len(trace)):
        if i == poi_list[poi_ctr]:
            op_trace.append(trace[i])
            poi_ctr += 1
            if poi_ctr == poi_max:
                break
    return op_trace

In [ ]:
poi_t1 = extract_poi(poi_list, traces1[1])
poi_t4 = extract_poi(poi_list, traces4[1])

poi_test = extract_poi(poi_list, trace1_avg)

In [ ]:
cw.plot(poi_t1) *  cw.plot(poi_t4) * cw.plot(flattened_poi1) * cw.plot(flattened_poi2) * cw.plot(flattened_poi3) * cw.plot(flattened_poi4)

In [ ]:
def std_dev_calc(mu, x_list, N):
    sum = 0
    for i in range(N):
        sum = sum + (x_list[i] - mu)**2
    return np.sqrt((1/N)*sum)

In [ ]:
std_dev_list = []
for s in range(5000-1):
    std_dev_list.append(std_dev_calc(trace1_avg[s],traces1[:,s],100))
np.asanyarray(std_dev_list).tofile("/home/40265864@ecit.qub.ac.uk/PhD/project_TOEPLITZ/results/std_dev_list_cw305.csv", sep=",")

In [ ]:
print(f"Captured {scope.adc.trig_count} segments")


In [ ]:
cw.plot(trace0 - trace1)    

TVLA Test

In [ ]:
from tqdm.autonotebook import trange

correct_key = gen_random_input()

N = 500
group1 = []
group2 = []
for i in trange(N, desc="Capturing Traces"):
    traces_cor = my_capture(correct_key)
    rand_key = gen_random_input()
    trace_rand = my_capture(rand_key)

    group1.append(traces_cor)
    if (rand_key != correct_key):
        group2.append(trace_rand)
    else:
        group1.append(trace_rand)

group1 = np.asarray(group1)
group2 = np.asarray(group2)

group1 = align_traces(group1)
group2 = align_traces(group2)

print(len(group1))
print(len(group2))

In [ ]:
mean1 = np.mean(group1, axis=0)
mean2 = np.mean(group2, axis=0)

cw.plot(mean2) * cw.plot(mean1)

In [ ]:
cw.plot(mean1 - mean2)

In [ ]:
from scipy.stats import ttest_ind
t_val = ttest_ind(group1, group2, axis=0, equal_var=False)[0]
cw.plot(t_val)

In [ ]:


import holoviews as hv
N = len(group1)
t_val = [ttest_ind(group1[:N//2], group2[:N//2], axis=0, equal_var=False)[0], 
         ttest_ind(group1[N//2:], group2[N//2:], axis=0, equal_var=False)[0]]

cv = cw.plot(t_val[0]) * cw.plot(t_val[1])
cv *= cw.plot([4.5]*len(group1[0]))
cv *= cw.plot([-4.5]*len(group1[0]))
cv



In [ ]:
#group1.tofile("/home/40265864@ecit.qub.ac.uk/PhD/project_TOEPLITZ/results/fpga_tvla_group1.csv", sep=",")
#group2.tofile("/home/40265864@ecit.qub.ac.uk/PhD/project_TOEPLITZ/results/fpga_tvla_group2.csv", sep=",")

np.savetxt("/home/40265864@ecit.qub.ac.uk/PhD/project_TOEPLITZ/results/fpga_tvla_group1.csv", group1, delimiter=",")
np.savetxt("/home/40265864@ecit.qub.ac.uk/PhD/project_TOEPLITZ/results/fpga_tvla_group2.csv", group2, delimiter=",")

In [ ]:
fig = cw.plot()
for t in trace:
    fig *= cw.plot(t)
fig

Artifical Noiseless Traces Experiment

In [10]:
def generate_noiseless_key(N):
    key = gen_random_input()
    traces = []
    for n in range(N):
        traces.append(my_capture(key))
    traces = align_traces(traces)
    trace = np.asarray(np.mean(traces, axis=0))
    return trace, key


In [11]:
def gen_template(key,L):
    traces = []
    for n in range(L):
        traces.append(my_capture(key))
    traces = align_traces(traces)
    trace = np.asarray(np.mean(traces, axis=0))
    return trace

In [12]:
from scipy import stats

In [19]:
def run_attack():
    N = 50
    L = 100
    target_trace, key = generate_noiseless_key(N)
    template1 = gen_template(data_bytes_key_in1,L)
    template2 = gen_template(data_bytes_key_in2,L)    
    template3 = gen_template(data_bytes_key_in3,L)    
    template4 = gen_template(data_bytes_key_in4,L)

    pc1 = stats.pearsonr(target_trace, template1)
    pc2 = stats.pearsonr(target_trace, template2)
    pc3 = stats.pearsonr(target_trace, template3)
    pc4 = stats.pearsonr(target_trace, template4)

    print(pc1)
    print(pc2)
    print(pc3)
    print(pc4)

    print(key[::1])





In [20]:
run_attack()

PearsonRResult(statistic=0.9583385558085609, pvalue=0.0)
PearsonRResult(statistic=0.9610975304237996, pvalue=0.0)
PearsonRResult(statistic=0.9652542570407125, pvalue=0.0)
PearsonRResult(statistic=0.9565779166899626, pvalue=0.0)
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


In [ ]:
#res = target.fpga_read(target.REG_CRYPT_CIPHEROUT,32)
res = target.readOutput()
print(res.hex())

In [ ]:
def get_traces(key, text):

    scope.arm()
    target.fpga_write(target.REG_CRYPT_KEY, key)
    target.fpga_write(target.REG_CRYPT_TEXTIN, text)    
    target.usb_trigger_toggle()
    target.go()
    ret = scope.capture()
    trace = scope.get_last_trace_segmented()
    return trace

In [ ]:
trace = get_traces(key, text)

In [ ]:
cw.plot(trace)

In [ ]:
#target.fpga_write(target.REG_CRYPT_KEY,key)
#target.fpga_write(target.REG_CRYPT_TEXTIN,text)


key_on_board = target.fpga_read(target.REG_CRYPT_KEY,32)
text_on_board = target.fpga_read(target.REG_CRYPT_TEXTIN,32)
output_on_board = target.fpga_read(target.REG_CRYPT_CIPHEROUT,32)


print("Before Set")
print(text_on_board.hex())
print(key_on_board.hex())
print(output_on_board.hex())



target.fpga_write(target.REG_CRYPT_TEXTIN,data_bytes_text_in)
target.fpga_write(target.REG_CRYPT_KEY,data_bytes_key_in)

scope.arm()

target.fpga_write(target.REG_CRYPT_GO,"\x01")
target.fpga_write(target.REG_CRYPT_GO,"\x00")

ret = scope.capture()
time.sleep(0.5)

key_on_board = target.fpga_read(target.REG_CRYPT_KEY,32)
text_on_board = target.fpga_read(target.REG_CRYPT_TEXTIN,32)
output_on_board = target.fpga_read(target.REG_CRYPT_CIPHEROUT,32)

print("\nAfter Set")
print(text_on_board.hex())
print(key_on_board.hex())
print(output_on_board.hex())


In [ ]:
def get_traces(key):

    scope.arm()
    target.fpga_write(target.REG_CRYPT_KEY, key)
    target.fpga_write(target.REG_USER_LED, [0x01])    
    #target.usb_trigger_toggle()
    target.go()
    ret = scope.capture()
    trace = scope.get_last_trace()
    target.fpga_write(target.REG_USER_LED, [0x00])    
    return trace



In [ ]:
target.fpga_write(target.REG_CRYPT_KEY, data_bytes_reset)
target.go()
res = target.readOutput()
print(res.hex())

In [ ]:
traces = get_traces(data_bytes_key_in)


In [ ]:
res = target.readOutput()
print(res.hex())

In [ ]:
cw.plot(traces)